# Gemma 2 Architecture (Short Summary)

## 1. Decoder-Only Transformer

Gemma 2 uses a standard autoregressive decoder-only Transformer architecture for next-token prediction.

---

## 2. RoPE (Rotary Position Embeddings)

Uses RoPE for positional encoding.

Benefits:
- better long-context handling
- stable attention
- improved extrapolation

---

## 3. GeGLU Activation

Uses GeGLU instead of standard GELU in feedforward layers.

Benefits:
- better gradient flow
- improved parameter efficiency
- stronger learning capability

---

## 4. Local Sliding Window Attention

Uses local attention with:

```text
window size = 4096
```

Tokens mainly attend to nearby tokens.

Benefits:
- lower memory usage
- faster inference
- scalable long contexts

---

## 5. Global Attention

Alternates between:
- local attention layers
- global attention layers

Global layers use full 8192-token context.

Benefits:
- preserves long-range reasoning
- improves global context understanding

---

## 6. Logit Soft-Capping

Caps logits using:

```text
logits = soft_cap * tanh(logits / soft_cap)
```

Benefits:
- stabilizes training
- prevents extremely large logits
- smoother optimization

---

## 7. RMSNorm

Uses RMSNorm instead of LayerNorm.

Benefits:
- simpler normalization
- lower computation cost
- improved stability

---

## 8. Grouped Query Attention (GQA)

Uses GQA with:

```text
num_groups = 2
```

Example:

```text
8 Query heads
2 shared KV groups
```

Benefits:
- smaller KV cache
- faster inference
- lower memory bandwidth

---

## 9. Large Vocabulary (256K)

Inherits Gemini’s 256K vocabulary.

Benefits:
- multilingual support
- better tokenization
- fewer token splits

---

# Main Idea

Gemma 2 combines:
- local + global attention
- efficient inference
- stable training
- strong parameter efficiency

to build practical high-performance open LLMs.

## Logit Soft-Capping

During attention and final prediction, Transformers compute values called:

```text
logits
```

These logits are passed into:

```text
softmax
```

to create probabilities.

---

### Problem

Sometimes logits become extremely large.

Example:

```text
[2, 5, 1000]
```

After softmax:

```text
[0, 0, 1]
```

The model becomes:
- overly confident
- unstable during training
- sensitive to small changes

Large logits can cause:
- unstable gradients
- exploding attention scores
- poor training stability

---

## Gemma 2 Solution

Gemma 2 limits logits using:

```text
logits = soft_cap * tanh(logits / soft_cap)
```

The:

```text
tanh()
```

function squashes values into a bounded range.

---

### Example

Suppose:

```text
soft_cap = 50
```

If logits become:

```text
1000
```

then:

```text
tanh(1000 / 50) ≈ 1
```

Final capped value becomes:

```text
50 × 1 = 50
```

So huge logits cannot explode infinitely.

---

## Why This Helps

Benefits:
- more stable training
- smoother optimization
- prevents extreme attention scores
- reduces numerical instability
- safer long-context training

---

## Simple Intuition

Without soft-capping:

```text
One token may dominate attention completely.
```

With soft-capping:

```text
Very large values are gently compressed.
```

So the model remains stable while still preserving relative importance.

# 3. Pre-training

## 3.1 Training Data

Gemma 2 was trained on massive datasets:

| Model | Training Tokens |
|---|---|
| 2B | 2 trillion |
| 9B | 8 trillion |
| 27B | 13 trillion |

Data sources include:
- web documents
- code
- scientific articles

The models are mainly English-focused and are not designed primarily for multilingual or multimodal tasks.

---

### Tokenizer

Gemma 2 uses the same tokenizer as:
- Gemma 1
- Gemini

It uses:
- SentencePiece tokenizer
- split digits
- preserved whitespace
- byte-level encoding

Vocabulary size:

```text
256K tokens
```

Benefits:
- multilingual support
- efficient tokenization
- fewer token splits

---

### Data Filtering

Gemma 2 filters training data to:
- remove unsafe content
- reduce sensitive information
- avoid benchmark contamination
- reduce memorization/recitation risks

This improves:
- safety
- privacy
- evaluation reliability

---

# 3.2 Knowledge Distillation

Gemma 2 uses knowledge distillation from a larger teacher model.

The teacher provides probabilities for the next token:

```text
P_teacher(x | context)
```

The smaller student model learns to imitate these probabilities.

---

### Simple Idea

Large model:

```text
teacher
```

smaller model:

```text
student
```

The student learns:
- how the teacher predicts tokens
- reasoning patterns
- probability distributions

instead of learning only from raw text.

---

### Benefits

- better small-model performance
- improved reasoning
- more efficient learning
- stronger parameter efficiency

This technique was also used in:
- Gemini 1.5

# 4. Post-Training

After pre-training, Gemma 2 is converted into an instruction-following assistant using several post-training stages.

---

## 1. Supervised Fine-Tuning (SFT)

The model is trained on:
- prompt-response examples
- synthetic + human-generated conversations

Most responses are generated by a larger teacher model.

This teaches the model:
- instruction following
- conversation style
- helpful responses

Gemma 2 also uses:
- behavioral cloning
- teacher distillation during SFT

Benefits:
- better reasoning
- improved assistant behavior
- stronger small-model performance

---

## 2. RLHF (Reinforcement Learning from Human Feedback)

After SFT, Gemma 2 applies RLHF.

Human preference data is used to train a:

```text
reward model
```

The reward model scores outputs based on:
- helpfulness
- safety
- conversational quality

The policy model is then optimized to maximize reward.

Benefits:
- safer responses
- fewer hallucinations
- better multi-turn conversations

---

## 3. Model Merging

Gemma 2 averages multiple trained models with different hyperparameters.

Benefits:
- more stable performance
- improved robustness
- better generalization

---

## 4. Data Filtering

Training data is filtered to remove:
- unsafe content
- toxic outputs
- personal information
- duplicated examples

The model is also trained to:
- hedge uncertain answers
- refuse unsafe requests
- improve factuality

Benefits:
- safer outputs
- reduced hallucinations
- better reliability

---

## 5. Formatting Improvements

Gemma 2 uses improved dialogue formatting with special control tokens.

Example:

```text
<end_of_turn><eos>
```

This helps:
- cleaner conversations
- better turn separation
- more stable chat behavior

---

# Main Goal of Post-Training

Post-training improves:
- instruction following
- safety
- conversational ability
- factuality
- helpfulness

while minimizing:
- hallucinations
- unsafe outputs
- toxic behavior

# Gemma 2 vs LLaMA vs GPT-3

| Feature | Gemma 2 | LLaMA | GPT-3 |
|---|---|---|---|
| Organization | Google DeepMind | Meta | OpenAI |
| Release | 2024 | 2023 | 2020 |
| Main Goal | practical efficient open LLM | efficient open LLM | massive few-shot LLM |
| Architecture | decoder-only Transformer | decoder-only Transformer | decoder-only Transformer |
| Attention | local + global attention | full causal attention | full causal attention |
| Positional Embedding | RoPE | RoPE | learned positional embeddings |
| Normalization | RMSNorm | RMSNorm | LayerNorm |
| Activation | GeGLU | SwiGLU | GELU |
| Context Length | 8K+ | 2K–4K initially | 2K |
| KV Cache Efficiency | optimized | standard | standard |
| GQA | yes | newer versions | no |
| Long Context Handling | very efficient | moderate | expensive |
| Parameter Efficiency | very strong | strong | lower |
| Training Focus | practical efficient models | efficient scaling | large-scale scaling |
| Inference Speed | fast | fast | slower |
| Open Weights | yes | yes | no |
| Main Innovation | local/global attention + efficient inference | better training efficiency | few-shot learning |

---
                    ┌──────────────────────┐
                    │   Pre-trained Model  │
                    │      (Gemma 2)       │
                    └──────────┬───────────┘
                               │
                               ▼
              ┌────────────────────────────────┐
              │ Supervised Fine-Tuning (SFT)   │
              │                                │
              │ • Prompt-response pairs        │
              │ • Synthetic + human data       │
              │ • Teacher-generated responses  │
              │ • Behavioral cloning           │
              └────────────────┬───────────────┘
                               │
                               ▼
              ┌────────────────────────────────┐
              │ Knowledge Distillation         │
              │                                │
              │ Teacher Model ──► Student      │
              │ Learns teacher probabilities   │
              │ and reasoning patterns         │
              └────────────────┬───────────────┘
                               │
                               ▼
              ┌────────────────────────────────┐
              │ RLHF                           │
              │ Reinforcement Learning from    │
              │ Human Feedback                 │
              │                                │
              │ • Human preference labels      │
              │ • Reward model                 │
              │ • Multi-turn optimization      │
              └────────────────┬───────────────┘
                               │
                               ▼
              ┌────────────────────────────────┐
              │ Data Filtering & Safety        │
              │                                │
              │ • Remove unsafe outputs        │
              │ • Remove sensitive info        │
              │ • Reduce hallucinations        │
              │ • Improve factuality           │
              └────────────────┬───────────────┘
                               │
                               ▼
              ┌────────────────────────────────┐
              │ Model Merging                  │
              │                                │
              │ Average multiple trained       │
              │ models with different          │
              │ hyperparameters                │
              └────────────────┬───────────────┘
                               │
                               ▼
                    ┌──────────────────────┐
                    │ Final Gemma 2        │
                    │ Instruction Model    │
                    └──────────────────────┘
# POST Training
## Supervised Fine-Tuning (SFT)

After pre-training, the model knows:
- grammar
- facts
- language patterns

But it still does not behave like a helpful assistant.

SFT teaches the model:

```text
How to respond to instructions and conversations
```
the response is generated by the teacher model.

---

# Step 1: Create Prompt-Response Data

Example:

| Prompt | Response |
|---|---|
| "Explain gravity" | "Gravity is a force..." |
| "Write Python code" | "def hello():" |

Data comes from:
- humans
- synthetic AI-generated examples

---

# Step 2: Teacher Model Generates Responses

A large powerful model acts as the:

```text
teacher
```

Example:

```text
User:
Explain photosynthesis.

Teacher:
Photosynthesis is the process by which plants convert sunlight into energy...
```

These teacher responses become training targets.

---

# Step 3: Student Model Learns

Gemma 2 (student model) receives:

```text
Prompt → predict teacher response
```

The model learns by minimizing prediction error.

Example:

```text
Input:
"Explain gravity"

Target:
"Gravity is the force..."
```

---

# Step 4: Behavioral Cloning

Behavioral cloning means:

```text
imitate expert behavior
```

The student copies:
- teacher writing style
- reasoning patterns
- conversational behavior
- instruction-following ability

---

# Why SFT Is Important

Pretraining teaches:

```text
language modeling
```

SFT teaches:

```text
assistant behavior
```

Without SFT:
- model may ignore instructions
- outputs may be messy
- conversations feel weak

With SFT:
- better instruction following
- cleaner responses
- more helpful conversations

---

# Simple Analogy

Pretraining:

```text
Learn the language
```

SFT:

```text
Learn how to be a helpful teacher/assistant
```


# 3. Gemma 2

Main idea:

```text
Practical efficient open models
```

Gemma 2 improves:
- long-context efficiency
- inference optimization
- hybrid local/global attention
- stability

Strengths:
- efficient inference
- scalable long contexts
- strong reasoning at smaller sizes

---

# Biggest Architectural Difference

## GPT-3

```text
Full attention everywhere
```

Very expensive for long sequences.

---

## LLaMA

```text
Efficiently trained full-attention model
```

Better than GPT-3 in efficiency.

---

## Gemma 2

```text
Hybrid local + global attention
```

Balances:
- efficiency
- reasoning quality
- long-context handling

---

# Philosophy Comparison

| Model | Philosophy |
|---|---|
| GPT-3 | scale bigger |
| LLaMA | train smarter |
| Gemma 2 | run smarter + practical efficiency |

---

# Final Summary

GPT-3 started the large-scale few-shot learning era, LLaMA showed that smaller efficiently trained open models can compete with huge systems, and Gemma 2 further improved efficiency using hybrid attention, optimized inference, and practical long-context design.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


# -------------------------
# RMSNorm
# -------------------------
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        return self.weight * x * torch.rsqrt(rms + self.eps)


# -------------------------
# GeGLU FeedForward
# -------------------------
class GeGLU(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.gate = nn.Linear(dim, hidden_dim)
        self.up = nn.Linear(dim, hidden_dim)
        self.down = nn.Linear(hidden_dim, dim)

    def forward(self, x):
        return self.down(F.gelu(self.gate(x)) * self.up(x))


# -------------------------
# Tiny Gemma 2 Attention
# local/global + GQA + soft-capping
# -------------------------
class TinyGemma2Attention(nn.Module):
    def __init__(
        self,
        dim=128,
        n_q_heads=8,
        n_kv_heads=2,
        local_window=16,
        global_attention=False,
        soft_cap=50.0
    ):
        super().__init__()

        self.dim = dim
        self.n_q_heads = n_q_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = dim // n_q_heads
        self.local_window = local_window
        self.global_attention = global_attention
        self.soft_cap = soft_cap

        self.q_proj = nn.Linear(dim, n_q_heads * self.head_dim)
        self.k_proj = nn.Linear(dim, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(dim, n_kv_heads * self.head_dim)
        self.o_proj = nn.Linear(dim, dim)

        self.group_size = n_q_heads // n_kv_heads

    def repeat_kv(self, x):
        return x.repeat_interleave(self.group_size, dim=1)

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_q_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)

        # GQA: fewer KV heads shared by many Q heads
        k = self.repeat_kv(k)
        v = self.repeat_kv(v)

        logits = q @ k.transpose(-2, -1)
        logits = logits / math.sqrt(self.head_dim)

        # logit soft-capping
        logits = self.soft_cap * torch.tanh(logits / self.soft_cap)

        # causal mask
        causal_mask = torch.tril(torch.ones(T, T, device=x.device))

        if self.global_attention:
            mask = causal_mask
        else:
            # local sliding window mask
            local_mask = torch.zeros(T, T, device=x.device)
            for i in range(T):
                start = max(0, i - self.local_window + 1)
                local_mask[i, start:i + 1] = 1
            mask = causal_mask * local_mask

        logits = logits.masked_fill(mask == 0, float("-inf"))

        attn = F.softmax(logits, dim=-1)
        out = attn @ v

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(out)


# -------------------------
# Transformer Block
# pre-norm + post-norm
# -------------------------
class TinyGemma2Block(nn.Module):
    def __init__(
        self,
        dim=128,
        n_q_heads=8,
        n_kv_heads=2,
        local_window=16,
        global_attention=False,
        mlp_hidden=256
    ):
        super().__init__()

        self.attn_pre_norm = RMSNorm(dim)
        self.attn = TinyGemma2Attention(
            dim=dim,
            n_q_heads=n_q_heads,
            n_kv_heads=n_kv_heads,
            local_window=local_window,
            global_attention=global_attention
        )
        self.attn_post_norm = RMSNorm(dim)

        self.ffn_pre_norm = RMSNorm(dim)
        self.ffn = GeGLU(dim, mlp_hidden)
        self.ffn_post_norm = RMSNorm(dim)

    def forward(self, x):
        attn_out = self.attn(self.attn_pre_norm(x))
        x = x + self.attn_post_norm(attn_out)

        ffn_out = self.ffn(self.ffn_pre_norm(x))
        x = x + self.ffn_post_norm(ffn_out)

        return x


# -------------------------
# Tiny Gemma 2 LM
# -------------------------
class TinyGemma2LM(nn.Module):
    def __init__(
        self,
        vocab_size=1000,
        dim=128,
        n_layers=6,
        n_q_heads=8,
        n_kv_heads=2,
        local_window=16,
        max_seq_len=128,
        final_soft_cap=30.0
    ):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(max_seq_len, dim)

        self.blocks = nn.ModuleList()

        for layer_id in range(n_layers):
            # alternate local and global attention
            global_attention = layer_id % 2 == 1

            self.blocks.append(
                TinyGemma2Block(
                    dim=dim,
                    n_q_heads=n_q_heads,
                    n_kv_heads=n_kv_heads,
                    local_window=local_window,
                    global_attention=global_attention,
                    mlp_hidden=dim * 4
                )
            )

        self.norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        self.final_soft_cap = final_soft_cap

    def forward(self, input_ids):
        B, T = input_ids.shape

        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)

        x = self.token_emb(input_ids) + self.pos_emb(pos)

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)

        logits = self.lm_head(x)

        # final logit soft-capping
        logits = self.final_soft_cap * torch.tanh(logits / self.final_soft_cap)

        return logits

In [5]:
# --------------------------------
# TRAINING
# --------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

model = TinyGemma2LM(
    vocab_size=100,
    dim=128,
    n_layers=6,
    n_q_heads=8,
    n_kv_heads=2,
    local_window=16,
    max_seq_len=128
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# --------------------------------
# Tiny character dataset
# --------------------------------

text = """
gemma 2 uses local and global attention.
gqa reduces kv cache memory.
rmsnorm stabilizes training.
"""

chars = sorted(list(set(text)))

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

vocab_size = len(chars)

data = torch.tensor(
    [stoi[c] for c in text],
    dtype=torch.long
)

# rebuild model with correct vocab size

model = TinyGemma2LM(
    vocab_size=vocab_size,
    dim=128,
    n_layers=6,
    n_q_heads=8,
    n_kv_heads=2,
    local_window=16,
    max_seq_len=128
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# --------------------------------
# batch loader
# --------------------------------

block_size = 32
batch_size = 8


def get_batch():
    ix = torch.randint(
        0,
        len(data) - block_size - 1,
        (batch_size,)
    )

    x = torch.stack([
        data[i:i+block_size]
        for i in ix
    ])

    y = torch.stack([
        data[i+1:i+block_size+1]
        for i in ix
    ])

    return x.to(device), y.to(device)


# --------------------------------
# training loop
# --------------------------------

for step in range(1000):

    x, y = get_batch()

    logits = model(x)

    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        y.reshape(-1)
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if step % 100 == 0:
        print(f"step {step} loss {loss.item():.4f}")

step 0 loss 3.3748
step 100 loss 0.1455
step 200 loss 0.0541
step 300 loss 0.0504
step 400 loss 0.0549
step 500 loss 0.0363
step 600 loss 0.0413
step 700 loss 0.0503
step 800 loss 0.0544
step 900 loss 0.0416


In [6]:
@torch.no_grad()
def generate(model, start_text, max_new_tokens=100):

    model.eval()

    input_ids = torch.tensor(
        [[stoi[c] for c in start_text]],
        dtype=torch.long
    ).to(device)

    for _ in range(max_new_tokens):

        x = input_ids[:, -block_size:]

        logits = model(x)

        next_token_logits = logits[:, -1, :]

        probs = F.softmax(next_token_logits, dim=-1)

        next_id = torch.multinomial(
            probs,
            num_samples=1
        )

        input_ids = torch.cat(
            [input_ids, next_id],
            dim=1
        )

    output = "".join([
        itos[i]
        for i in input_ids[0].tolist()
    ])

    return output

In [7]:
# This is a toy educational Gemma 2-style model, not the real Google Gemma 2.
print(generate(model, "gemma", 100))

gemma 2 uses local and global attention.
gqa reduces kv cache memory.
rmsnorm stabilizes training.
g.
gqa
